In [ ]:
def main(datasources, start_date, end_date):
    """
    微价格与成交 VWAP 背离。

    经济含义：比较盘口隐含的多层 microprice 与当前分钟成交 VWAP，并按分钟价格波动范围归一化。
    可能有效的理由：目标本身是未来 30 分钟 VWAP；盘口隐含价格高于已成交 VWAP 时，可能存在尚未反映到成交价格的潜在压力，属于盘口与成交的跨源背离。

    评测时平台会注入 datasources、start_date 和 end_date。
    返回且仅返回 ['date', 'instrument', 'factor'] 三列。
    """
    import math
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    def minute_micro_vwap_gap(
        ask_price1: float,
        ask_price2: float,
        ask_price3: float,
        ask_price4: float,
        ask_price5: float,
        bid_price1: float,
        bid_price2: float,
        bid_price3: float,
        bid_price4: float,
        bid_price5: float,
        ask_volume1: float,
        ask_volume2: float,
        ask_volume3: float,
        ask_volume4: float,
        ask_volume5: float,
        bid_volume1: float,
        bid_volume2: float,
        bid_volume3: float,
        bid_volume4: float,
        bid_volume5: float,
        high: float,
        low: float,
        amount: float,
        volume: float,
    ) -> float:
        """多层 microprice 相对当前分钟成交 VWAP 的有界背离。"""
        ask_p = (ask_price1, ask_price2, ask_price3, ask_price4, ask_price5)
        bid_p = (bid_price1, bid_price2, bid_price3, bid_price4, bid_price5)
        ask_v = (ask_volume1, ask_volume2, ask_volume3, ask_volume4, ask_volume5)
        bid_v = (bid_volume1, bid_volume2, bid_volume3, bid_volume4, bid_volume5)
        total = sum(ask_v) + sum(bid_v)
        cross_value = sum(p * v for p, v in zip(ask_p, bid_v))
        cross_value += sum(p * v for p, v in zip(bid_p, ask_v))
        micro = cross_value / (total + 1e-8)
        trade_vwap = amount / (volume + 1e-8)
        scale = max(high - low, ask_price1 - bid_price1, 1e-6)
        return float(math.tanh((micro - trade_vwap) / scale))

    def label_time_segment(value) -> int:
        """按官方示例的左开右闭规则返回固定的 30 分钟截面。"""
        hhmmss = value.hour * 10000 + value.minute * 100 + value.second
        sections = (
            (90000, 93000), (93000, 100000), (100000, 103000),
            (103000, 110000), (110000, 113000), (130000, 133000),
            (133000, 140000), (140000, 143000),
        )
        for section_start, section_end in sections:
            if section_start < hhmmss <= section_end:
                return section_end
        return -1

    sql = f"""
    WITH cte_snapshot AS (
        SELECT
            date,
            instrument_id,
            strftime(date, '%Y-%m-%d') AS trading_day,
            minute_micro_vwap_gap(
                COALESCE(ask_price1, 0),
                COALESCE(ask_price2, 0),
                COALESCE(ask_price3, 0),
                COALESCE(ask_price4, 0),
                COALESCE(ask_price5, 0),
                COALESCE(bid_price1, 0),
                COALESCE(bid_price2, 0),
                COALESCE(bid_price3, 0),
                COALESCE(bid_price4, 0),
                COALESCE(bid_price5, 0),
                COALESCE(ask_volume1, 0),
                COALESCE(ask_volume2, 0),
                COALESCE(ask_volume3, 0),
                COALESCE(ask_volume4, 0),
                COALESCE(ask_volume5, 0),
                COALESCE(bid_volume1, 0),
                COALESCE(bid_volume2, 0),
                COALESCE(bid_volume3, 0),
                COALESCE(bid_volume4, 0),
                COALESCE(bid_volume5, 0),
                COALESCE(high, 0),
                COALESCE(low, 0),
                COALESCE(amount, 0),
                COALESCE(volume, 0)
            ) AS minute_signal,
            label_time_segment(date) AS time_segment
        FROM {bar1m}
        WHERE time_segment != -1
    ),
    cte_window AS (
        SELECT trading_day, time_segment, instrument_id,
               AVG(minute_signal) AS factor
        FROM cte_snapshot
        GROUP BY instrument_id, trading_day, time_segment
    )
    SELECT CAST(CONCAT(
        f.trading_day, ' ',
        strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
    ) AS DATETIME) AS date,
    all_instruments.instrument,
    f.factor AS factor
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
        udf_list=[
                        dai.DaiUDF(name="minute_micro_vwap_gap", function=minute_micro_vwap_gap),

            dai.DaiUDF(name="label_time_segment", function=label_time_segment),
        ],
    ).df()

    df = df.replace([np.inf, -np.inf], np.nan)
    instruments_table = datasources.get("instruments", "cpt_jyc_2026_instruments")
    stk_pool = dai.query(
        f"SELECT date, instrument FROM {instruments_table}",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    result = pd.merge(df, stk_pool, how="inner", on=["date", "instrument"])
    result = result.drop_duplicates(["date", "instrument"])
    return result[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)
